# Notebook 1: Data Inspection and Subject Filtering

Starts from the full ABIDE-I phenotypic table, filters down to the 1035 valid subjects, merges in TR/volume counts, drops subjects on FD and scan-duration criteria, and writes the filtered phenotypic CSV.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")
PHENOTYPIC_PATH = DATA_DIR / "metadata" / "Phenotypic_V1_0b_preprocessed1.csv"
TR_PATH = DATA_DIR / "subject_tr.csv"
OUTPUT_PATH = DATA_DIR / "phenotypic_filtered.csv"

FD_THRESHOLD = 0.5          # mm, keep <= this
WINDOW_SECONDS = 96
N_WINDOWS = 3
MIN_DURATION_SECONDS = WINDOW_SECONDS * N_WINDOWS  # 288

## Step 1-2: Load the full phenotypic table and filter to the 1035 valid subjects

Same `no_filename` exclusion used for downloading (`FILE_ID` empty or equal to `no_filename` is dropped).

In [2]:
df_full = pd.read_csv(PHENOTYPIC_PATH, encoding="utf-8-sig")
print(f"Full phenotypic table: {len(df_full)} rows")

file_id = df_full["FILE_ID"].astype(str).str.strip()
valid_mask = (file_id != "") & (file_id.str.lower() != "no_filename")

df = df_full.loc[valid_mask].copy()
df["FILE_ID"] = file_id.loc[valid_mask]

print(f"After no_filename exclusion: {len(df)} rows")
assert len(df) == 1035, f"Expected 1035 subjects, got {len(df)}"

Full phenotypic table: 1112 rows
After no_filename exclusion: 1035 rows


## Step 3: Merge with TR / volume-count data

From `scripts/extract_tr.py`'s output (`subject_tr.csv`), read directly from each subject's `func_preproc` NIfTI header.

In [3]:
tr_df = pd.read_csv(TR_PATH)[["FILE_ID", "TR_seconds", "N_VOLUMES"]]

df = df.merge(tr_df, on="FILE_ID", how="left", validate="one_to_one")

missing_tr = df["TR_seconds"].isna().sum()
print(f"Merged with TR data: {len(df)} rows, {missing_tr} missing TR")
assert missing_tr == 0, "All 1035 subjects should have a TR value"

Merged with TR data: 1035 rows, 0 missing TR


## Step 4: Drop by FD (keep `func_mean_fd <= 0.5` mm)

A subject with a missing `func_mean_fd` value cannot be verified against the threshold, so it is dropped too.

In [4]:
df["func_mean_fd"] = pd.to_numeric(df["func_mean_fd"], errors="coerce")

fd_pass = df["func_mean_fd"] <= FD_THRESHOLD
n_dropped_fd = (~fd_pass).sum()

print(f"Dropped for FD > {FD_THRESHOLD} (or missing): {n_dropped_fd}")
df = df.loc[fd_pass].copy()
print(f"Remaining: {len(df)}")

Dropped for FD > 0.5 (or missing): 30
Remaining: 1005


## Step 5: Drop by scan duration (need >= 3 x 96s = 288s)

In [5]:
df["scan_duration_seconds"] = df["N_VOLUMES"] * df["TR_seconds"]

duration_pass = df["scan_duration_seconds"] >= MIN_DURATION_SECONDS
n_dropped_duration = (~duration_pass).sum()

print(f"Dropped for scan_duration_seconds < {MIN_DURATION_SECONDS}: {n_dropped_duration}")
df = df.loc[duration_pass].copy()
print(f"Remaining: {len(df)}")

Dropped for scan_duration_seconds < 288: 26
Remaining: 979


## Step 6: Write the filtered phenotypic CSV

In [6]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(df)} subjects to {OUTPUT_PATH}")
print(f"\nFunnel: 1035 -> {1035 - n_dropped_fd} (post-FD) -> {len(df)} (post-duration)")
df[["FILE_ID", "SITE_ID", "DX_GROUP", "TR_seconds", "N_VOLUMES", "func_mean_fd", "scan_duration_seconds"]].head()

Wrote 979 subjects to ../data/raw/phenotypic_filtered.csv

Funnel: 1035 -> 1005 (post-FD) -> 979 (post-duration)


,FILE_ID,SITE_ID,DX_GROUP,TR_seconds,N_VOLUMES,func_mean_fd,scan_duration_seconds
0,Pitt_0050003,PITT,1,1.5,196,0.322092,294.0
1,Pitt_0050004,PITT,1,1.5,196,0.127745,294.0
2,Pitt_0050005,PITT,1,1.5,196,0.128136,294.0
3,Pitt_0050006,PITT,1,1.5,196,0.070143,294.0
4,Pitt_0050007,PITT,1,1.5,196,0.151246,294.0
